# **Xenium data**


In [ ]:
!pip install spatialdata spatialdata-io


In [ ]:
!pip install xenium-sdk scanpy squidpy seaborn matplotlib


In [ ]:
!pip install squidpy


In [ ]:
import spatialdata as sd
from spatialdata_io import xenium

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import squidpy as sq

In [ ]:
xenium_path = "./Xenium"
zarr_path = "./Xenium.zarr"

In [ ]:
import os
print(os.getcwd())
print(os.listdir("."))


In [ ]:
from spatialdata_io import xenium


In [ ]:
pip install spatialdata-io


In [ ]:
import spatialdata_io
print(spatialdata_io.__version__)


In [ ]:
import os
print(os.getcwd())


In [ ]:
import os
print(os.listdir("/content"))


In [ ]:
!wget https://cf.10xgenomics.com/samples/xenium/2.0.0/Xenium_V1_human_Lung_2fov/Xenium_V1_human_Lung_2fov_outs.zip


In [ ]:
import zipfile

with zipfile.ZipFile("Xenium_V1_human_Lung_2fov_outs.zip", "r") as z:
    z.extractall("/content")


In [ ]:
import os
print(os.listdir("/content"))



In [ ]:
xenium_path = "/content/Xenium_V1_human_Lung_2fov_outs"


In [ ]:
xenium_path = "/content"


In [ ]:
xenium_path = "./Xenium"
zarr_path = "./Xenium.zarr"

In [ ]:
from spatialdata_io import xenium

sdata = xenium(xenium_path)


In [ ]:
sdata = xenium(xenium_path)

In [ ]:
adata = sdata.tables["table"]
adata

In [ ]:
adata.obs

In [ ]:
adata.obsm["spatial"]

In [ ]:
cprobes = (
    adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
cwords = (
    adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
print(f"Negative DNA probe count % : {cprobes}")
print(f"Negative decoding count % : {cwords}")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total counts per cell")
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])

axs[1].set_title("Transcript counts per cell")
sns.histplot(adata.obs["transcript_counts"], kde=False, ax=axs[1])

axs[2].set_title("Cell area")
sns.histplot(adata.obs["cell_area"], kde=False, ax=axs[2])

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3],
)


In [ ]:
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)


In [ ]:
pip install leidenalg igraph


In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata)

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "total_counts",
        "leiden",
    ],
    wspace=0.4,
)


In [ ]:
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color=[
        "leiden",
    ],
    wspace=0.4,
)

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)

In [ ]:
sq.gr.centrality_scores(adata, cluster_key="leiden")

In [ ]:
sq.pl.centrality_scores(adata, cluster_key="leiden", figsize=(16, 5))

In [ ]:
sdata.tables["subsample"] = sc.pp.subsample(adata, fraction=0.5, copy=True)

In [ ]:
adata_subsample = sdata.tables["subsample"]

In [ ]:
sq.gr.co_occurrence(
    adata_subsample,
    cluster_key="leiden",
)
sq.pl.co_occurrence(
    adata_subsample,
    cluster_key="leiden",
    clusters="12",
    figsize=(10, 10),
)
sq.pl.spatial_scatter(
    adata_subsample,
    color="leiden",
    shape=None,
    size=2,
)

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="leiden")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 7))
sq.pl.nhood_enrichment(
    adata,
    cluster_key="leiden",
    figsize=(8, 8),
    title="Neighborhood enrichment adata",
    ax=ax[0],
)
sq.pl.spatial_scatter(adata_subsample, color="leiden", shape=None, size=2, ax=ax[1])

In [ ]:
sq.gr.spatial_neighbors(adata_subsample, coord_type="generic", delaunay=True)
sq.gr.spatial_autocorr(
    adata_subsample,
    mode="moran",
    n_perms=100,
    n_jobs=1,
)
adata_subsample.uns["moranI"].head(10)

In [ ]:
sq.pl.spatial_scatter(
    adata_subsample,
    library_id="spatial",
    color=[
        "AREG",
        "MET",
    ],
    shape=None,
    size=2,
    img=False,
)

In [ ]:
import spatialdata_plot

gene_name = ["AREG", "MET"]
for name in gene_name:
    sdata.pl.render_images("morphology_focus").pl.render_shapes(
        "cell_circles",
        color=name,
        table_name="table",
        use_raw=False,
    ).pl.show(
        title=f"{name} expression over Morphology image",
        coordinate_systems="global",
        figsize=(10, 5),
    )

In [ ]:
xenium_path = "/content"
from spatialdata_io import xenium
sdata = xenium(xenium_path)

from napari_spatialdata import Interactive

Interactive(sdata)